In [1]:
import subprocess   # For launching the analysis engine
import win32com.client
from win32com.client import gencache

# win32com.client.constants.* come from the makepy type-library cache, so generate it for the two
# LARSA libraries this example uses. Then force late binding so the objects keep their real
# method/property names when reading data from LARSA (an early-bound cache hides element fields 
# behind base COM interfaces).
gencache.EnsureDispatch("LarsaData.clsUnits")
gencache.EnsureDispatch("LarsaElements.clsProject")
gencache.EnsureDispatch("LarsaElements.clsAnalysisResults")
gencache.GetClassForCLSID = lambda clsid: None  

def CreateObject(class_name):
    return win32com.client.Dispatch(class_name)

def columnHeaders(analysis, dataType):
    # Results spreadsheet headers (with units).
    try:
        info = win32com.client.Record("RESULTS_SPREAD_INFO", analysis.ResultGroups)
        info.rq.dataType = dataType
        info.display = 0
        return [h.replace("\r\n", " ") for h in analysis.getSpreadsheetColumnHeaders(info, [])[1] if isinstance(h, str)]
    except Exception:
        # pywin32 312 has an issue where writing any field of a COM struct raises TypeError 
        # (pywin32 <= 311 returns the labelled headers as expected).
        return []

# ----- Generate the model --------------------------------
project = CreateObject("LarsaElements.clsProject")

# Set units for the 9 units categories:
#   Input Data -> Units: 0 = MATERIAL_UNITS,  1 = Section_UNITS,  2 = Spring_units,  
#       3 = COORDINATE_UNITS, 4 = LOAD_UNITS,  5 = mass_units;
#   Results -> Units: 6 = DISPLACEMENT_UNITS,  7 = FORCE_UNITS,  8 = STRESS_UNITS.
for category in range(9):
    project.units.SetUnits(category, True, 
        project.units.UnitFromString(win32com.client.constants.UNIT_LENGTH, "in"), 
        project.units.UnitFromString(win32com.client.constants.UNIT_FORCE, "kip"), 
        project.units.UnitFromString(win32com.client.constants.UNIT_TEMPERATURE, "F"), 
        project.units.UnitFromString(win32com.client.constants.UNIT_ANGLE, "rad"))

material = CreateObject("LarsaElements.clsMaterial")
material.Name = "Steel"
material.modulusOfElasticity = 29000.0    
material.unitWeight = 0.000284
project.materials.append(material)


ModuleNotFoundError: No module named 'win32com'